# Vertex AI Gemini 2.5 Flash with Google AI Search Test

Testing Gemini 2.5 Flash with Google AI search embedded via Vertex AI API.


In [ ]:
%pip install google-generativeai python-dotenv pillow requests

import google.generativeai as genai
import os
from dotenv import load_dotenv
from PIL import Image
import requests
from io import BytesIO

load_dotenv()


  Using cached google_generativeai-0.8.5-py3-none-any.whl.metadata (3.9 kB)
  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached google_api_core-2.28.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached google_api_python_client-2.187.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached google_auth-2.43.0-py2.py3-none-any.whl.metadata (6.6 kB)
  Using cached proto_plus-1.26.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached protobuf-5.29.5-cp38-abi3-macosx_10_9_universal2.whl.metadata (592 bytes)
  Using cached googleapis_common_protos-1.72.0-py3-none-any.whl.metadata (9.4 kB)
  Using cached grpcio_status-1.76.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached cachetools-6.2.1-py3-none-any.whl.metadata (5.5 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached rsa-4.9.1-py3-none-any.whl.metadata (5.6 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible w

True

In [2]:
# API configuration
api_key = os.getenv("GOOGLE_AI_API_KEY") or os.getenv("VERTEX_AI_API_KEY")
model_name = "gemini-2.5-flash"

if not api_key:
    print("❌ GOOGLE_AI_API_KEY or VERTEX_AI_API_KEY not found in environment")
    print("\nPlease set the API key:")
    print("export GOOGLE_AI_API_KEY=your_key_here")
    print("\nOr add to .env file:")
    print("GOOGLE_AI_API_KEY=your_key_here")
else:
    print(f"✅ API key found")
    print(f"Testing model: {model_name}")


❌ GOOGLE_AI_API_KEY or VERTEX_AI_API_KEY not found in environment

Please set the API key:
export GOOGLE_AI_API_KEY=your_key_here

Or add to .env file:
GOOGLE_AI_API_KEY=your_key_here


In [ ]:
# Initialize client
if api_key:
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel(model_name)
    print("✅ Client initialized")
else:
    model = None


In [ ]:
# Simple test prompt
prompt = "Say 'Gemini 2.5 Flash is working!' and explain what you're best at."

print(f"Prompt: {prompt}")
print("\n" + "="*70)


In [ ]:
# Generate response
if model:
    try:
        response = model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                temperature=0.7,
                max_output_tokens=300,
            )
        )
        
        print("✅ Response:")
        print(response.text)
        
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\nCheck:")
        print("1. API key is valid")
        print("2. Model name is correct: gemini-2.5-flash")
        print("3. You have API access enabled")
else:
    print("⚠️  Model not initialized. Check API key.")


In [ ]:
# Test with Google AI Search (grounding)
search_prompt = """What are the latest developments in AI agent frameworks in 2025? 
Please provide recent information with sources."""

print("Testing Google AI Search integration...")
print(f"\nPrompt: {search_prompt}")
print("\n" + "="*70)

if model:
    try:
        # Enable grounding with Google Search
        response = model.generate_content(
            search_prompt,
            generation_config=genai.types.GenerationConfig(
                temperature=0.7,
                max_output_tokens=500,
            ),
            tools=[{"google_search": {}}]  # Enable Google Search grounding
        )
        
        print("✅ Response with Search:")
        print(response.text)
        
        # Check if grounding metadata is available
        if hasattr(response, 'grounding_metadata'):
            print("\n📚 Grounding Sources:")
            if response.grounding_metadata:
                print(response.grounding_metadata)
        
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\nNote: Google Search grounding may require additional configuration")
        print("Trying without grounding...")
        
        # Fallback without grounding
        try:
            response = model.generate_content(
                search_prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.7,
                    max_output_tokens=500,
                )
            )
            print("\n✅ Response (without grounding):")
            print(response.text)
        except Exception as e2:
            print(f"❌ Fallback error: {e2}")


## Multimodal Capabilities

Gemini 2.5 Flash supports multimodal inputs including images, text, and more. Let's test image analysis capabilities.


In [ ]:
# Test 1: Image analysis from URL
image_url = "https://storage.googleapis.com/generativeai-downloads/images/scones.jpg"
image_prompt = "What's in this image? Describe it in detail."

print("Testing image analysis from URL...")
print(f"Image URL: {image_url}")
print(f"Prompt: {image_prompt}")
print("\n" + "="*70)

if model:
    try:
        # Download image from URL
        response_img = requests.get(image_url)
        img = Image.open(BytesIO(response_img.content))
        
        # Generate content with image and text
        response = model.generate_content(
            [image_prompt, img],
            generation_config=genai.types.GenerationConfig(
                temperature=0.7,
                max_output_tokens=500,
            )
        )
        
        print("✅ Response:")
        print(response.text)
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  Model not initialized. Check API key.")


In [ ]:
# Test 2: Image analysis from local file (if available)
# You can upload an image file to test this
local_image_path = None  # Set to your image path, e.g., "path/to/image.jpg"

if local_image_path and os.path.exists(local_image_path):
    print("Testing image analysis from local file...")
    print(f"Image path: {local_image_path}")
    print("\n" + "="*70)
    
    if model:
        try:
            # Load local image
            img = Image.open(local_image_path)
            
            # Generate content with image and text
            response = model.generate_content(
                ["Describe this image in detail. What do you see?", img],
                generation_config=genai.types.GenerationConfig(
                    temperature=0.7,
                    max_output_tokens=500,
                )
            )
            
            print("✅ Response:")
            print(response.text)
            
        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("⚠️  Model not initialized. Check API key.")
else:
    print("ℹ️  No local image path provided. Set 'local_image_path' variable to test with local images.")
    print("   Example: local_image_path = 'path/to/your/image.jpg'")


In [ ]:
# Test 3: Multiple images with text
image_urls = [
    "https://storage.googleapis.com/generativeai-downloads/images/scones.jpg",
    # Add more image URLs if needed
]

multimodal_prompt = """Compare these images. What are the similarities and differences? 
If there's only one image, describe it in detail and suggest what it could be used for."""

print("Testing multiple images with text...")
print(f"Number of images: {len(image_urls)}")
print(f"Prompt: {multimodal_prompt}")
print("\n" + "="*70)

if model and image_urls:
    try:
        # Download and prepare images
        images = []
        for url in image_urls:
            try:
                response_img = requests.get(url)
                img = Image.open(BytesIO(response_img.content))
                images.append(img)
                print(f"✅ Loaded image from: {url}")
            except Exception as e:
                print(f"⚠️  Failed to load image from {url}: {e}")
        
        if images:
            # Combine text prompt with images
            content = [multimodal_prompt] + images
            
            # Generate content
            response = model.generate_content(
                content,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.7,
                    max_output_tokens=800,
                )
            )
            
            print("\n✅ Response:")
            print(response.text)
        else:
            print("❌ No images were successfully loaded")
            
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
else:
    if not model:
        print("⚠️  Model not initialized. Check API key.")
    else:
        print("⚠️  No image URLs provided.")


In [ ]:
# Test 4: Image + Text Q&A
image_url = "https://storage.googleapis.com/generativeai-downloads/images/scones.jpg"
qa_prompt = """Look at this image and answer:
1. What is the main subject?
2. What colors are prominent?
3. What could this image be used for?
4. Write a creative caption for this image."""

print("Testing image Q&A...")
print(f"Image URL: {image_url}")
print(f"Prompt: {qa_prompt}")
print("\n" + "="*70)

if model:
    try:
        # Download image
        response_img = requests.get(image_url)
        img = Image.open(BytesIO(response_img.content))
        
        # Generate content
        response = model.generate_content(
            [qa_prompt, img],
            generation_config=genai.types.GenerationConfig(
                temperature=0.8,  # Slightly higher for creative caption
                max_output_tokens=600,
            )
        )
        
        print("✅ Response:")
        print(response.text)
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  Model not initialized. Check API key.")


In [ ]:
# Test 5: Image analysis with specific task (e.g., OCR, object detection)
image_url = "https://storage.googleapis.com/generativeai-downloads/images/scones.jpg"
task_prompt = """Analyze this image and provide:
- A detailed description
- Any text visible in the image (OCR)
- Objects or items you can identify
- The style or type of photography
- Any interesting details"""

print("Testing detailed image analysis...")
print(f"Image URL: {image_url}")
print("\n" + "="*70)

if model:
    try:
        # Download image
        response_img = requests.get(image_url)
        img = Image.open(BytesIO(response_img.content))
        
        # Generate content
        response = model.generate_content(
            [task_prompt, img],
            generation_config=genai.types.GenerationConfig(
                temperature=0.5,  # Lower temperature for more factual analysis
                max_output_tokens=700,
            )
        )
        
        print("✅ Response:")
        print(response.text)
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  Model not initialized. Check API key.")
